# 🔗 Outskill Masterclass — AI-Assisted Coding Lab

A hands-on notebook for the **AI-Assisted Coding with Claude Code** session. You will use a single
**OpenRouter** key (the same one you use for Claude Code) to try out prompt-engineering patterns,
AI code review, debugging, documentation, and prompt chaining — all against one small demo repo.

**How to use it:** run the cells top to bottom. Each cell builds on the one before. Cells that call
the model are marked with 💬 and make one API call each, so run only what you need during the class.

**What you will do**
1. Connect to OpenRouter
2. Load the demo repo (`SnipURL`, a tiny FastAPI URL shortener — with a few planted bugs)
3. Try 4 core prompt-engineering patterns
4. Run an AI code review
5. Reproduce a real bug, then fix it with the model
6. Generate a README, an architecture diagram, and an API reference
7. Compare one big prompt vs. a chain of small ones
8. Do a few exercises on your own

## 1 · Setup

OpenRouter is OpenAI-compatible, so we use the official `openai` client and just point it at
OpenRouter's URL. **Recommended:** add your key to Colab Secrets (the 🔑 icon in the left sidebar)
as `OPENROUTER_KEY` — then nothing is typed or saved in the notebook. Otherwise you will be prompted
to paste it (input is hidden).

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from getpass import getpass

# 1) Try Colab Secrets first (add a secret named OPENROUTER_KEY).
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata

    OPENROUTER_API_KEY = userdata.get("OPENROUTER_KEY")
except Exception:
    pass

# 2) Fall back to a hidden prompt.
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass("Paste your OpenRouter API key (input hidden): ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={"X-Title": "SnipURL Masterclass"},
)

# Change this to any model id from https://openrouter.ai/models
MODEL = "anthropic/claude-sonnet-4.5"
print("Client ready. Using model:", MODEL)

Paste your OpenRouter API key (input hidden): ··········
Client ready. Using model: anthropic/claude-sonnet-4.5


In [ ]:
def ask(
    prompt,
    system="You are a helpful senior software engineer.",
    model=None,
    max_tokens=1200,
):
    """Send one prompt to the model through OpenRouter and return the reply text."""
    try:
        resp = client.chat.completions.create(
            model=model or MODEL,
            max_tokens=max_tokens,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
        )
        return resp.choices[0].message.content
    except Exception as e:
        return (
            "[API ERROR] "
            + str(e)
            + "\n\nIf this looks like a model error, set MODEL to a valid id "
            "from https://openrouter.ai/models"
        )

💬 **Smoke test** — confirm your key works before the class moves on.

In [ ]:
print(ask("hey what's up"))

Hey! Not much, just here and ready to help. 👋

What's up with you? Got any code you're working on, architecture questions, debugging nightmares, or anything else I can help with?


## 2 · The demo repo: `SnipURL`

A tiny FastAPI URL shortener, embedded below as strings so there is nothing to clone. It has **no
README, no docstrings, no tests** — and a few **deliberate bugs** — so the rest of the lab has real
work to do. Skim it now; you do not need to memorize it.

In [ ]:
REPO = {
    "models.py": """from pydantic import BaseModel


class ShortenRequest(BaseModel):
    url: str


class ShortenResponse(BaseModel):
    code: str
    short_url: str


class StatsResponse(BaseModel):
    code: str
    long_url: str
    clicks: int
""",
    "storage.py": """# In-memory store. Swap for a real database (Redis or Postgres) in production.
class Storage:
    def __init__(self):
        self._urls = {}      # code -> long url
        self._clicks = {}    # code -> click count
        self._counter = 0

    def next_id(self):
        self._counter += 1
        return self._counter

    def save(self, code, long_url):
        self._urls[code] = long_url
        self._clicks[code] = 0

    def get_url(self, code):
        return self._urls.get(code)

    def increment_clicks(self, code):
        count = self._clicks.get(code, 0)
        count + 1  # BUG: result is thrown away; should be self._clicks[code] = count + 1
        return self._clicks[code]

    def get_clicks(self, code):
        return self._clicks.get(code, 0)
""",
    "service.py": """import string

ALPHABET = string.ascii_lowercase + string.ascii_uppercase + string.digits  # 62 characters


def encode_base62(num):
    out = ""
    while num > 0:
        out = ALPHABET[num % 62] + out
        num //= 62
    return out  # BUG: encode_base62(0) returns "" instead of a valid one-character code


def record_visit(code, log=[]):  # BUG: mutable default argument is shared across all calls
    log.append(code)
    return log


def shorten(storage, long_url):
    new_id = storage.next_id()
    code = encode_base62(new_id)
    storage.save(code, long_url)
    return code
""",
    "main.py": """from fastapi import FastAPI
from fastapi.responses import RedirectResponse

from models import ShortenRequest, ShortenResponse, StatsResponse
from storage import Storage
from service import shorten, record_visit

app = FastAPI(title="SnipURL", version="0.1.0")
store = Storage()


@app.post("/shorten", response_model=ShortenResponse)
def create_short_url(req: ShortenRequest):
    code = shorten(store, req.url)  # NOTE: no validation that req.url is a real URL
    return ShortenResponse(code=code, short_url=f"http://snip.url/{code}")


@app.get("/{code}")
def redirect_to_url(code: str):
    long_url = store.get_url(code)
    store.increment_clicks(code)
    record_visit(code)
    return RedirectResponse(long_url)  # BUG: unknown code -> long_url is None; should return 404


@app.get("/stats/{code}", response_model=StatsResponse)
def get_stats(code: str):
    return StatsResponse(
        code=code,
        long_url=store.get_url(code),
        clicks=store.get_clicks(code),
    )
""",
}


def repo_text():
    parts = []
    for name, src in REPO.items():
        parts.append("# ===== " + name + " =====\n" + src)
    return "\n\n".join(parts)


REPO_TEXT = repo_text()
print("Loaded SnipURL:", ", ".join(REPO.keys()))

Loaded SnipURL: models.py, storage.py, service.py, main.py


## 3 · Prompt-engineering patterns

Four patterns you will reuse all session. Run each and read the two outputs side by side — the
*difference* is the lesson.

### 3.1 Be specific (vague vs. specific) 💬
A vague ask gets a vague answer. Constraints focus the model on what you actually care about.

In [ ]:
vague = ask("Review this code:\n\n" + REPO["storage.py"])

specific = ask(
    "Review storage.py below. List ONLY concrete bugs. For each give: the method name, "
    "what is wrong, and the one-line fix. Ignore style.\n\n" + REPO["storage.py"]
)

print("--- VAGUE PROMPT ---\n", vague)
print("\n\n--- SPECIFIC PROMPT ---\n", specific)

--- VAGUE PROMPT ---
 Good catch on the bug! Here's my code review:

## 🐛 **Critical Bug**

**Line 18: `increment_clicks` doesn't actually increment**
```python
count + 1  # Result is discarded!
```
**Fix:**
```python
self._clicks[code] = count + 1
```
Or more idiomatically:
```python
self._clicks[code] = self._clicks.get(code, 0) + 1
```

---

## 🔍 **Other Issues**

### 1. **Race Conditions (Thread Safety)**
If multiple requests arrive simultaneously:
- `next_id()` could return duplicate IDs
- `increment_clicks()` could lose counts (even after fixing the bug)

**Solution:** Add locking
```python
from threading import Lock

class Storage:
    def __init__(self):
        self._urls = {}
        self._clicks = {}
        self._counter = 0
        self._lock = Lock()

    def next_id(self):
        with self._lock:
            self._counter += 1
            return self._counter

    def increment_clicks(self, code):
        with self._lock:
            self._clicks[code] = self._clicks.ge

### 3.2 Role / persona priming 💬
Telling the model *who to be* shifts what it notices.

In [ ]:
neutral = ask("Tell me about this endpoint file:\n\n" + REPO["main.py"])

security = ask(
    "Tell me about this endpoint file:\n\n" + REPO["main.py"],
    system="You are a paranoid application-security reviewer. You care most about missing "
    "input validation, error handling, and unsafe responses.",
)

print("--- NEUTRAL ROLE ---\n", neutral)
print("\n\n--- SECURITY ROLE ---\n", security)

--- NEUTRAL ROLE ---
 # URL Shortener Endpoint File Analysis

This is a FastAPI application implementing a URL shortener service (like bit.ly). Here's a breakdown:

## **Architecture**

- **Framework**: FastAPI with Pydantic models for request/response validation
- **Storage**: Custom `Storage` class handling persistence
- **Service Layer**: Separate `service` module with business logic

## **Endpoints**

### 1. `POST /shorten` - Create Short URL
```python
Input: ShortenRequest (contains original URL)
Output: ShortenResponse (short code + full short URL)
```
- Generates a short code for the given URL
- Returns formatted short URL (e.g., `http://snip.url/abc123`)
- **⚠️ Issue**: No URL validation (noted in comment)

### 2. `GET /{code}` - Redirect to Original URL
```python
Input: Short code from URL path
Output: HTTP redirect to original URL
```
- Retrieves original URL from storage
- Increments click counter
- Records visit analytics
- **🐛 Bug**: Returns 500 error instead of 404 when c

### 3.3 Structured output 💬
When you will reuse the result, specify the exact shape. Here we ask for a Markdown table and
render it inline.

In [ ]:
from IPython.display import Markdown, display

table = ask(
    "From the FastAPI app below, produce a Markdown table of every endpoint with columns: "
    "Method, Path, Request body, Response, Notes. Output ONLY the table.\n\n"
    + REPO["main.py"]
)
display(Markdown(table))

| Method | Path | Request body | Response | Notes |
|--------|------|--------------|----------|-------|
| POST | /shorten | ShortenRequest | ShortenResponse | No validation that req.url is a real URL |
| GET | /{code} | None | RedirectResponse | BUG: unknown code → long_url is None; should return 404 |
| GET | /stats/{code} | None | StatsResponse | None |

### 3.4 Self-critique loop 💬
Ask the model to review and improve its own first answer. Cheap, and usually a real upgrade.

In [ ]:
draft = ask(
    "Write a one-paragraph description of what SnipURL does, based on:\n\n" + REPO_TEXT
)

improved = ask(
    "Here is a draft description:\n\n"
    + draft
    + "\n\nCritique it for accuracy and vagueness, then output an improved final version only."
)

print("--- DRAFT ---\n", draft)
print("\n\n--- AFTER SELF-CRITIQUE ---\n", improved)

--- DRAFT ---
 SnipURL is a URL shortening service built with FastAPI that converts long URLs into compact, shareable links using base62 encoding. When a user submits a URL via the `/shorten` endpoint, the service generates a unique short code, stores the mapping in memory, and returns a shortened URL (e.g., `http://snip.url/abc123`). When someone visits a short link, the service redirects them to the original URL while tracking click counts, and users can retrieve statistics (original URL and click count) for any short code via the `/stats` endpoint. The current implementation uses in-memory storage and includes several bugs: click counts aren't properly incremented, visiting a non-existent code causes a crash instead of returning a 404, the base62 encoder fails for ID zero, and the visit logging function has a mutable default argument issue.


--- AFTER SELF-CRITIQUE ---
 # Final Version

SnipURL is a URL shortening service built with FastAPI that converts long URLs into compact, sha

## 4 · AI code review 💬

Now point a senior-reviewer persona at the whole repo and ask for specific, concrete findings. It
should surface the planted bugs (mutable default arg, the click counter, the missing 404, the
base62 edge case, and the missing URL validation).

In [ ]:
review = ask(
    "You are reviewing the SnipURL repo below. Find correctness bugs, missing error handling, and "
    "security issues. For each finding give: file, the problem, and a concrete fix. Be specific and "
    "do not invent issues.\n\n" + REPO_TEXT,
    system="You are a senior backend engineer doing a thorough pull-request review.",
    max_tokens=1600,
)
print(review)

## 5 · Debugging — reproduce, then fix

The best debugging prompts start from a *reproduction*. The cell below actually runs the repo's
`Storage` class so you can see the bug with your own eyes: clicks stay at 0.

In [ ]:
# Load the real Storage class straight from the repo string and watch the bug happen.
exec(REPO["storage.py"], globals())

s = Storage()
s.save("abc", "https://example.com")
s.increment_clicks("abc")
s.increment_clicks("abc")
print(
    "Clicks after 2 visits (should be 2):", s.get_clicks("abc")
)  # prints 0 -> the bug

💬 Now hand the model the reproduction and ask it to reason step by step before fixing.

In [ ]:
fix = ask(
    "The class below should count clicks, but get_clicks() always returns 0 after "
    "increment_clicks() is called. Think step by step about why, then give the corrected method.\n\n"
    + REPO["storage.py"],
    system="You are debugging Python. Reason step by step before giving the fix.",
)
print(fix)

**Verify the fix yourself:** edit the `increment_clicks` line in the Section 2 repo cell (or the
`exec` demo cell above) to `self._clicks[code] = count + 1`, re-run it and the reproduction cell —
you should now see `2`. Seeing your fix turn the test green is the whole point.

## 6 · Documentation: README + architecture + API reference

This is the headline use case: point the model at a repo and generate the docs nobody wrote.

### 6.1 README 💬

In [ ]:
readme = ask(
    "Write a clear README.md for the SnipURL project below. Include: a one-line summary, features, "
    "how to install and run it (it is a FastAPI app), an example request for each endpoint, and the "
    "project structure. Output Markdown only.\n\n" + REPO_TEXT,
    max_tokens=1600,
)
display(Markdown(readme))

### 6.2 Architecture diagram (Mermaid) 💬

In [ ]:
mermaid = ask(
    'Generate a Mermaid "flowchart LR" diagram showing how a request flows through SnipURL: '
    "client -> FastAPI endpoint (main.py) -> service.py -> storage.py. Output only a fenced mermaid "
    "code block.\n\n" + REPO_TEXT
)
print(mermaid)
print(
    "\nTip: paste this into https://mermaid.live or a GitHub .md file to see it render."
)

### 6.3 API reference (structured output) 💬

In [ ]:
api_ref = ask(
    "Produce an API reference for SnipURL as Markdown. For each endpoint include method, path, "
    "parameters, request body schema, response schema, and status codes. Use the models and routes "
    "below as the source of truth.\n\n" + REPO_TEXT,
    max_tokens=1600,
)
display(Markdown(api_ref))

## 7 · The main event: one big prompt vs. a chain

Same goal, two strategies. Run both and compare.

### 7.1 Naive — everything in one prompt 💬
It tends to do each task shallowly: misses bugs, and the docs may describe the *buggy* behavior.

In [ ]:
naive = ask(
    "Review this repo, fix all the bugs, write docstrings, and generate a README. "
    "Do it all in one response.\n\n" + REPO_TEXT,
    max_tokens=1800,
)
print(naive)

### 7.2 Chained — small steps, each feeding the next 💬💬💬💬
Understand → find bugs → fix the bugs that were found → document the *corrected* code. Notice how
step 3 receives step 2's bug list, and step 4 documents step 3's fixed code — so the docs describe
correct behavior, not bugs.

In [ ]:
# Step 1: understand
summary = ask(
    "Summarize what each file in this repo does, in 4-5 lines.\n\n" + REPO_TEXT
)

# Step 2: find the bugs (this output feeds step 3)
bugs = ask(
    "List every correctness bug in this repo as a numbered list. One line each: file + problem.\n\n"
    + REPO_TEXT
)

# Step 3: fix exactly the bugs found in step 2 (we pass the bug list back in)
fixes = ask(
    "Here is a confirmed bug list for the repo:\n\n"
    + bugs
    + "\n\nOutput the corrected code for each affected file. Code only, grouped by filename.\n\n"
    + REPO_TEXT,
    max_tokens=1800,
)

# Step 4: document the now-corrected code (we pass the fixes, not the original)
final_readme = ask(
    "Using this corrected code, write a short, accurate README (summary, endpoints, run steps).\n\n"
    + fixes,
    max_tokens=1400,
)

print("STEP 1 - SUMMARY\n", summary)
print("\n\nSTEP 2 - BUG LIST\n", bugs)
print("\n\nSTEP 3 - FIXES (preview)\n", fixes[:1200], "...")
print("\n\nSTEP 4 - README FROM CORRECTED CODE\n", final_readme)

**Takeaway:** chaining is not "more prompts for the sake of it." It is *ordering the work* so each
step stands on solid ground — find bugs before you fix them, fix them before you document, document
correct code instead of bugs. This is exactly what Claude Code's **Plan Mode** and **subagents**
automate for you.

## 8 · Your turn

1. **Write your own reviewer.** Fill in the system prompt below to focus *only* on security, then run it.
2. **Add a chain step.** After step 3 above, add a step that asks the model to write `pytest` tests for the fixed code.
3. **Force a schema.** Change the API-reference cell to output valid **OpenAPI JSON** instead of Markdown.
4. **Swap the model.** Set `MODEL` to a different id from [openrouter.ai/models](https://openrouter.ai/models) and re-run the review — compare results. (Tip: simpler/free models often miss bugs or break structured output, which is itself a useful lesson.)

In [ ]:
# EXERCISE 1 - write your own reviewer persona, then run it.
my_system = ""  # <- describe a security-focused reviewer here

print(
    ask(
        "Review this file:\n\n" + REPO["main.py"],
        system=my_system or "You are a code reviewer.",
    )
)

---
### Recap
- The model is only as good as the **context** and **constraints** you give it.
- **Reproduce** bugs before asking for fixes; **verify** fixes by re-running.
- **Chain** prompts so each step builds on a correct previous one.

**Go deeper:** Claude Code docs (Plan Mode, subagents, CLAUDE.md) and the prompt-engineering guide at
docs.claude.com.